# Ingest completed monthly history

This notebook discovers each completed MMSDM month, downloads its `DISPATCHREGIONSUM` archive into Bronze and extracts the CSV. The current month is excluded because its monthly archive is not complete yet.

### Storage prerequisite

Before running the pipeline, I created the Databricks Volume `/Volumes/workspace/default/aemo_mlops_volume`. The ingestion notebooks write their Bronze files there, and the Silver transformation reads from the same location.

## 1. Find the file for one month

MMSDM uses a year/month folder structure. The function opens that folder and selects the `DISPATCHREGIONSUM` ZIP while excluding similarly named pre-dispatch files.

### How the monthly filename pattern works

```python
r'[^"\s<>]*DISPATCHREGIONSUM[^"\s<>]*\.zip'
```

- `r'...'` makes this a raw Python string, so backslashes are passed directly to the pattern matcher.
- `[^"\s<>]*` accepts any number of filename characters, but stops at a quote, whitespace or an HTML `<` or `>` boundary.
- `DISPATCHREGIONSUM` requires the table name we need. Text may appear before and after it because MMSDM filenames also contain archive metadata and dates.
- `\.zip` requires the literal `.zip` extension. The dot is escaped because an unescaped dot means “any character” in a pattern.

The search ignores letter case. A separate check then removes `PREDISPATCH` matches, because those contain forecasts rather than the observed dispatch history used here.

In [0]:
import os
import re
import time
import zipfile
from datetime import date
from urllib.parse import urljoin

import requests


def build_monthly_url(year, month):
    # Build the folder used by AEMO for this completed year and month.
    folder = (
        "https://www.nemweb.com.au/"
        "Data_Archive/Wholesale_Electricity/MMSDM/"
        f"{year}/MMSDM_{year}_{month:02d}/"
        "MMSDM_Historical_Data_SQLLoader/DATA/"
    )

    # Read the HTML directory listing so the notebook does not depend on a hard-coded filename.
    response = requests.get(folder)

    # A missing folder means AEMO has not published that month in this layout.
    if response.status_code == 404:
        return None

    response.raise_for_status()

    # Keep ZIP links for DISPATCHREGIONSUM, the source containing TOTALDEMAND.
    files = re.findall(
        r'[^"\s<>]*DISPATCHREGIONSUM[^"\s<>]*\.zip',
        response.text,
        flags=re.IGNORECASE
    )

    # PREDISPATCH files are forecasts, not the observed dispatch history built here.
    files = [
        file for file in files
        if "PREDISPATCH" not in file.upper()
    ]

    if not files:
        return None

    # Convert the relative filename from the listing into a complete download URL.
    return urljoin(folder, files[0])

### Check one month

This quick call confirms that the folder parser returns the expected archive URL before scanning the full history.

In [0]:
url = build_monthly_url(2015, 1)
print(url)

## 2. Discover all completed months

The scan begins in 2015, follows the same lookup for every month and stops before the current month.

In [0]:
today = date.today()

urls = []

start = time.time()

for year in range(2015, today.year + 1):
    for month in range(1, 13):

        # The monthly source is only used once a complete month has been published.
        if (year, month) >= (today.year, today.month):
            continue

        month_start = time.time()

        print(f"Checking {year}-{month:02d}...", end=" ")

        url = build_monthly_url(year, month)

        elapsed = time.time() - month_start

        if url:
            urls.append(url)
            print(f"found ({elapsed:.1f}s)")
        else:
            print(f"not published ({elapsed:.1f}s)")

total_time = time.time() - start

print()
print(f"Found {len(urls)} files")
print(f"Total time: {total_time:.1f} seconds")

In [0]:
# Display the discovered URLs for a simple visual check before downloading.
urls

## 3. Download new archives into Bronze

Bronze preserves the source ZIPs. Existing files are skipped, which makes repeated runs safe and avoids unnecessary downloads.

In [0]:
def download_if_not_exists(url, bronze_folder):
    filename = url.split("/")[-1]
    path = os.path.join(bronze_folder, filename)

    # Re-running the notebook must not download a source file twice.
    if os.path.exists(path):
        print(f"Skipping: {filename}")
        return path

    print(f"Downloading: {filename}")
    start = time.time()

    response = requests.get(url)
    response.raise_for_status()

    # Store the original bytes unchanged in the monthly Bronze folder.
    with open(path, "wb") as file:
        file.write(response.content)

    print(
        f"Finished: {filename} - "
        f"{time.time() - start:.1f} seconds"
    )

    return path

In [0]:
bronze_folder = "/Volumes/workspace/default/aemo_mlops_volume/bronze/monthly"

os.makedirs(bronze_folder, exist_ok=True)

for url in urls:
    download_if_not_exists(url, bronze_folder)

## 4. Extract the monthly CSV files

Each monthly archive contains the table CSV used by the Silver transformation. Extraction is also skipped when every expected file already exists.

In [0]:
csv_folder = bronze_folder + "_uncompressed"

os.makedirs(csv_folder, exist_ok=True)

zip_files = [
    file for file in os.listdir(bronze_folder)
    if file.endswith(".zip")
]

start = time.time()

print(f"Found {len(zip_files)} ZIP files\n")

for i, filename in enumerate(sorted(zip_files), 1):

    path = os.path.join(bronze_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:

        files = zip_file.namelist()

        # Treat the archive as complete only when all of its members are present.
        already_extracted = all(
            os.path.exists(os.path.join(csv_folder, file))
            for file in files
        )

        if already_extracted:
            print(f"[{i}/{len(zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(zip_files)}] Extracting: {filename}")

        zip_file.extractall(csv_folder)

print()
print(f"Finished in {time.time() - start:.1f} seconds")